In [1]:
# === Import des librairies ===
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# === 1. Charger le fichier CSV ===
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv",sep=";")  # Remplace par le chemin correct
print("Aperçu des données :")
display(df.head())

# === 2. Sélection des colonnes numériques ===
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Inclure la target si ce n'est pas déjà le cas
if 'Montant' not in numeric_columns:
    numeric_columns.append('Montant')

# === 3. Winsorisation sur toutes les colonnes numériques ===
df_winsor = df.copy()
for col in numeric_columns:
    # Définir les bornes à 5% et 95% (modifiable)
    df_winsor[col] = winsorize(df_winsor[col], limits=[0.05, 0.05])

# === 4. Vérification après winsorisation ===
print("Statistiques après winsorisation :")
display(df_winsor[numeric_columns].describe())


Aperçu des données :


,dataloadingdate,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,JourSemaine,ISIN,Libellé,Nombre de Titres,Montant,Echéance,Taux
0,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",1459.0,1.500,62.0,7.6
1,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",291.0,0.300,183.0,7.5
2,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,50000.0,5.000,13.0,8.6
3,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,10000.0,1.000,7.0,8.6
4,01/07/2025,1,7,27,3,1,Tuesday,TN0008000606,"BTA 6,7% Avril 2028",5660.0,5.742,31.0,9.1


Statistiques après winsorisation :


,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,Nombre de Titres,Montant,Echéance,Taux
count,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000
mean,14.850200,6.600730,26.628509,2.550155,1.954409,7597.774165,4.133039,33.962599,8.627887
std,9.057001,3.394496,14.508804,1.103134,1.474442,10679.660190,4.635870,36.140342,0.759614
min,1.000000,1.000000,3.000000,1.000000,0.000000,281.000000,0.225000,6.000000,7.250000
25%,7.000000,4.000000,14.000000,2.000000,1.000000,1078.000000,1.000000,10.000000,8.030000
50%,14.000000,7.000000,27.000000,3.000000,2.000000,3180.000000,2.001000,21.000000,8.970000
75%,23.000000,10.000000,40.000000,4.000000,3.000000,8986.500000,5.500000,32.000000,9.010000
max,30.000000,12.000000,49.000000,4.000000,4.000000,41689.000000,18.000000,145.000000,9.890000


In [2]:
# === 5. Séparation des features et de la target ===
X = df_winsor.drop(columns=['Montant'])  # Toutes les colonnes sauf la target
y = df_winsor['Montant']                 # Target

# === 6. Encodage des variables catégorielles si besoin ===
# Exemple : convertir les colonnes de type object en variables dummy
X = pd.get_dummies(X, drop_first=True)

# === 7. Séparation en train et test ===
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === 8. Vérification des dimensions ===
print(f"Dimensions X_train : {X_train.shape}")
print(f"Dimensions X_test  : {X_test.shape}")
print(f"Dimensions y_train : {y_train.shape}")
print(f"Dimensions y_test  : {y_test.shape}")


Dimensions X_train : (16213, 1330)
Dimensions X_test  : (4054, 1330)
Dimensions y_train : (16213,)
Dimensions y_test  : (4054,)


In [3]:
# === Linear Regression ===
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# 1) Linear Regression brute
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_brut = lin_reg.predict(X_test)
rmse_brut = np.sqrt(mean_squared_error(y_test, y_pred_brut))
print(f"Linear Regression RMSE brut          : {rmse_brut:.4f}")

# 2) Linear Regression normalisée (features standardisées)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lin_reg_norm = LinearRegression()
lin_reg_norm.fit(X_train_scaled, y_train)
y_pred_norm = lin_reg_norm.predict(X_test_scaled)
rmse_norm = np.sqrt(mean_squared_error(y_test, y_pred_norm))
print(f"Linear Regression RMSE normalisé     : {rmse_norm:.4f}")

# 3) Linear Regression avec Feature Selection (top 6 corrélations)
# Calcul des corrélations uniquement sur les colonnes numériques de X_train
numeric_cols_train = X_train.select_dtypes(include=[np.number]).columns
correlations = X_train[numeric_cols_train].corrwith(y_train).abs().sort_values(ascending=False)
top_features = correlations.index[:6].tolist()

lin_reg_feat = LinearRegression()
lin_reg_feat.fit(X_train[top_features], y_train)
y_pred_feat = lin_reg_feat.predict(X_test[top_features])
rmse_feat = np.sqrt(mean_squared_error(y_test, y_pred_feat))
print(f"Linear Regression RMSE feat select   : {rmse_feat:.4f}")
print(f"Top features : {top_features}")


Linear Regression RMSE brut          : 312781.2677
Linear Regression RMSE normalisé     : 2133167013426.2729
Linear Regression RMSE feat select   : 3.6606
Top features : ['Nombre de Titres', 'Taux', 'JourSemaineNum', 'Trimestre', 'Echéance', 'Mois']


In [11]:
# === Ridge Regression ===
from sklearn.linear_model import Ridge

# 1) Ridge Regression brute
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train, y_train)
y_pred_brut = ridge.predict(X_test)
rmse_brut = np.sqrt(mean_squared_error(y_test, y_pred_brut))
print(f"Ridge RMSE brut          : {rmse_brut:.4f}")

# 2) Ridge Regression normalisée
ridge_norm = Ridge(alpha=1.0, random_state=42)
ridge_norm.fit(X_train_scaled, y_train)
y_pred_norm = ridge_norm.predict(X_test_scaled)
rmse_norm = np.sqrt(mean_squared_error(y_test, y_pred_norm))
print(f"Ridge RMSE normalisé     : {rmse_norm:.4f}")

# 3) Ridge Regression avec Feature Selection (mêmes top features que Linear Regression)
ridge_feat = Ridge(alpha=1.0, random_state=42)
ridge_feat.fit(X_train[top_features], y_train)
y_pred_feat = ridge_feat.predict(X_test[top_features])
rmse_feat = np.sqrt(mean_squared_error(y_test, y_pred_feat))
print(f"Ridge RMSE feat select   : {rmse_feat:.4f}")


Ridge RMSE brut          : 2.7142
Ridge RMSE normalisé     : 2.7364
Ridge RMSE feat select   : 3.6606


In [5]:
# === Lasso Regression ===
from sklearn.linear_model import Lasso

# 1) Lasso Regression brute
lasso = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso.fit(X_train, y_train)
y_pred_brut = lasso.predict(X_test)
rmse_brut = np.sqrt(mean_squared_error(y_test, y_pred_brut))
print(f"Lasso RMSE brut          : {rmse_brut:.4f}")

# 2) Lasso Regression normalisée
lasso_norm = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso_norm.fit(X_train_scaled, y_train)
y_pred_norm = lasso_norm.predict(X_test_scaled)
rmse_norm = np.sqrt(mean_squared_error(y_test, y_pred_norm))
print(f"Lasso RMSE normalisé     : {rmse_norm:.4f}")

# 3) Lasso Regression avec Feature Selection (top_features)
lasso_feat = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso_feat.fit(X_train[top_features], y_train)
y_pred_feat = lasso_feat.predict(X_test[top_features])
rmse_feat = np.sqrt(mean_squared_error(y_test, y_pred_feat))
print(f"Lasso RMSE feat select   : {rmse_feat:.4f}")



Lasso RMSE brut          : 3.6506
Lasso RMSE normalisé     : 2.7913
Lasso RMSE feat select   : 3.6670


In [6]:
# === Random Forest Regressor ===
from sklearn.ensemble import RandomForestRegressor

# 1) Random Forest brut
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf_brut = rf.predict(X_test)
rmse_rf_brut = np.sqrt(mean_squared_error(y_test, y_pred_rf_brut))
print(f"Random Forest RMSE brut          : {rmse_rf_brut:.4f}")

# 2) Random Forest normalisé
rf_norm = RandomForestRegressor(n_estimators=100, random_state=42)
rf_norm.fit(X_train_scaled, y_train)
y_pred_rf_norm = rf_norm.predict(X_test_scaled)
rmse_rf_norm = np.sqrt(mean_squared_error(y_test, y_pred_rf_norm))
print(f"Random Forest RMSE normalisé     : {rmse_rf_norm:.4f}")

# 3) Random Forest avec Feature Selection
rf_feat = RandomForestRegressor(n_estimators=100, random_state=42)
rf_feat.fit(X_train[top_features], y_train)
y_pred_rf_feat = rf_feat.predict(X_test[top_features])
rmse_rf_feat = np.sqrt(mean_squared_error(y_test, y_pred_rf_feat))
print(f"Random Forest RMSE feat select   : {rmse_rf_feat:.4f}")


Random Forest RMSE brut          : 0.9807
Random Forest RMSE normalisé     : 0.9811
Random Forest RMSE feat select   : 1.9286


In [7]:
# === Gradient Boosting Regressor ===
from sklearn.ensemble import GradientBoostingRegressor

# 1) Gradient Boosting brut
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb_brut = gb.predict(X_test)
rmse_gb_brut = np.sqrt(mean_squared_error(y_test, y_pred_gb_brut))
print(f"Gradient Boosting RMSE brut          : {rmse_gb_brut:.4f}")

# 2) Gradient Boosting normalisé
gb_norm = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_norm.fit(X_train_scaled, y_train)
y_pred_gb_norm = gb_norm.predict(X_test_scaled)
rmse_gb_norm = np.sqrt(mean_squared_error(y_test, y_pred_gb_norm))
print(f"Gradient Boosting RMSE normalisé     : {rmse_gb_norm:.4f}")

# 3) Gradient Boosting avec Feature Selection
gb_feat = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_feat.fit(X_train[top_features], y_train)
y_pred_gb_feat = gb_feat.predict(X_test[top_features])
rmse_gb_feat = np.sqrt(mean_squared_error(y_test, y_pred_gb_feat))
print(f"Gradient Boosting RMSE feat select   : {rmse_gb_feat:.4f}")


Gradient Boosting RMSE brut          : 1.8230
Gradient Boosting RMSE normalisé     : 1.8229
Gradient Boosting RMSE feat select   : 2.4409


In [8]:
# === Winsorisation de toutes les features et de la target ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.stats.mstats import winsorize

# df doit contenir toutes les données, incluant la target 'Montant'
# Sélection des colonnes numériques
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Winsorisation des colonnes numériques
df_winsor = df.copy()
for col in numeric_columns:
    # Winsorisation aux 5% et 95%
    df_winsor[col] = winsorize(df[col], limits=[0.05, 0.05])

# Séparation features / target
X = df_winsor.drop(columns=['Montant'])
y = df_winsor['Montant']

# Split train/test
X_train_winsor, X_test_winsor, y_train_winsor, y_test_winsor = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Vérification des dimensions
print("Dimensions X_train :", X_train_winsor.shape)
print("Dimensions X_test  :", X_test_winsor.shape)
print("Dimensions y_train :", y_train_winsor.shape)
print("Dimensions y_test  :", y_test_winsor.shape)


Dimensions X_train : (16213, 12)
Dimensions X_test  : (4054, 12)
Dimensions y_train : (16213,)
Dimensions y_test  : (4054,)


In [13]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# === Identification des types de colonnes ===
# Séparer les colonnes numériques et catégorielles
numeric_features = X_train_winsor.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_winsor.select_dtypes(include=['object']).columns.tolist()

# === Préprocessing ===
# Transformer pour gérer les variables catégorielles et numériques
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# === Pipeline avec preprocessing + Ridge ===
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state=42))
])

# === Définition de la plage d'hyperparamètres ===
param_dist_ridge = {
    'regressor__alpha': [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]
}

# === Création du RandomizedSearchCV ===
random_search_ridge = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist_ridge,
    n_iter=10,  # nombre de combinaisons à tester pour gagner du temps
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1
)

# === Fit sur les données d'entraînement ===
random_search_ridge.fit(X_train_winsor, y_train)

# === Meilleur modèle ===
best_ridge = random_search_ridge.best_estimator_

# === Prédiction sur le test ===
y_pred_ridge = best_ridge.predict(X_test_winsor)

# === Calcul RMSE ===
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print(f"Ridge RMSE fine-tuning rapide : {rmse_ridge:.4f}")
print(f"Meilleur alpha : {best_ridge.named_steps['regressor'].alpha}")

c:\Users\zizou\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 9 is smaller than n_iter=10. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Ridge RMSE fine-tuning rapide : 2.6832
Meilleur alpha : 10


In [15]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# === Identification des types de colonnes ===
# Séparer les colonnes numériques et catégorielles
numeric_features = X_train_winsor.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_winsor.select_dtypes(include=['object']).columns.tolist()

# === Préprocessing ===
# Transformer pour gérer les variables catégorielles et numériques
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# === Pipeline avec preprocessing + Lasso ===
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(random_state=42, max_iter=10000))
])

# === Lasso Regression Fine-Tuning ===
param_dist_lasso = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100]
}

# === RandomizedSearchCV ===
random_search_lasso = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist_lasso,
    n_iter=7,  # égal au nombre de valeurs
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1
)

# === Fit sur les données d'entraînement winsorisées ===
random_search_lasso.fit(X_train_winsor, y_train)

# === Meilleur modèle ===
best_lasso = random_search_lasso.best_estimator_

# === Prédiction sur le test ===
y_pred_lasso = best_lasso.predict(X_test_winsor)

# === Calcul RMSE ===
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

print(f"Lasso RMSE fine-tuning rapide : {rmse_lasso:.4f}")
print(f"Meilleur alpha : {best_lasso.named_steps['regressor'].alpha}")

Lasso RMSE fine-tuning rapide : 2.6801
Meilleur alpha : 0.001


In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# === Identification des types de colonnes ===
# Séparer les colonnes numériques et catégorielles
numeric_features = X_train_winsor.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_winsor.select_dtypes(include=['object']).columns.tolist()

# === Préprocessing ===
# Transformer pour gérer les variables catégorielles (Random Forest n'a pas besoin de standardisation)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# === Pipeline avec preprocessing + Random Forest ===
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# === Random Forest Fine-Tuning Rapide ===
param_dist_rf = {
    'regressor__n_estimators': [100, 200, 300, 400, 500],
    'regressor__max_depth': [10, 20, 30, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': [0.6, 0.8, 1.0]
}

# === RandomizedSearchCV ===
random_search_rf = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist_rf,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1
)

# === Fit sur les données d'entraînement winsorisées ===
random_search_rf.fit(X_train_winsor, y_train)

# === Meilleur modèle ===
best_rf = random_search_rf.best_estimator_

# === Prédiction sur le test ===
y_pred_rf = best_rf.predict(X_test_winsor)

# === Calcul RMSE ===
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"Random Forest RMSE fine-tuning rapide : {rmse_rf:.4f}")
print(f"Meilleurs hyperparamètres RF : {random_search_rf.best_params_}")

Random Forest RMSE fine-tuning rapide : 0.9606
Meilleurs hyperparamètres RF : {'regressor__n_estimators': 500, 'regressor__min_samples_split': 5, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 0.6, 'regressor__max_depth': None}


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# === Gradient Boosting Fine-Tuning Rapide ===
param_dist_gb = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.6, 0.8, 1.0]
}

# Garder uniquement les colonnes numériques
X_train_num = X_train_winsor.select_dtypes(include=[np.number])
X_test_num  = X_test_winsor.select_dtypes(include=[np.number])

gb_model = GradientBoostingRegressor(random_state=42)
random_search_gb = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_dist_gb,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1
)

# Fit sur les données d'entraînement numériques
random_search_gb.fit(X_train_num, y_train)

# Meilleur modèle
best_gb = random_search_gb.best_estimator_

# Prédiction sur le test
y_pred_gb = best_gb.predict(X_test_num)

# Calcul RMSE
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

print(f"Gradient Boosting RMSE fine-tuning rapide : {rmse_gb:.4f}")
print(f"Meilleurs hyperparamètres GB : {random_search_gb.best_params_}")


ValueError: 
All the 150 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\zizou\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\zizou\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zizou\anaconda3\Lib\site-packages\sklearn\ensemble\_gb.py", line 659, in fit
    X, y = self._validate_data(
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zizou\anaconda3\Lib\site-packages\sklearn\base.py", line 650, in _validate_data
    X, y = check_X_y(X, y, **c

In [ ]:
# ==============================
# Vérification et sauvegarde de la DataFrame winsorisée
# ==============================

import pandas as pd

# Vérifier les colonnes numériques
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Colonnes numériques winsorisées :", numeric_columns)

# Afficher quelques statistiques pour vérifier la winsorisation
print(df[numeric_columns].describe())

# Enregistrer la DataFrame winsorisée dans le même fichier
df.to_csv("dataframefinale.csv", index=False)
print("La DataFrame winsorisée a été enregistrée dans 'dataframefinale.csv'.")



In [20]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import numpy as np

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_winsor)
X_test_scaled  = scaler.transform(X_test_winsor)

# Hyperparamètres à tester
ridge_params = {'alpha': np.logspace(-2, 3, 20)}

# RandomizedSearchCV
ridge_search = RandomizedSearchCV(
    Ridge(random_state=42),
    param_distributions=ridge_params,
    n_iter=20,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42
)
ridge_search.fit(X_train_scaled, y_train)

# Meilleur modèle et RMSE
ridge_best = ridge_search.best_estimator_
y_pred_ridge = ridge_best.predict(X_test_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print(f"Ridge RMSE fine-tuning avancé : {rmse_ridge:.4f}")
print(f"Meilleur alpha : {ridge_best.alpha}")


ValueError: could not convert string to float: '15/07/2024'

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# Hyperparamètres à tester pour Lasso
lasso_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100]
}

# RandomizedSearchCV pour Lasso
lasso_search = RandomizedSearchCV(
    Lasso(random_state=42, max_iter=10000),
    param_distributions=lasso_params,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42
)

# Fit sur les données winsorisées
lasso_search.fit(X_train_winsor, y_train)

# Meilleur modèle et RMSE
lasso_best = lasso_search.best_estimator_
y_pred_lasso = lasso_best.predict(X_test_winsor)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

print(f"Lasso RMSE fine-tuning avancé : {rmse_lasso:.4f}")
print(f"Meilleur alpha : {lasso_search.best_params_['alpha']}")


In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Supposons que df est déjà votre dataframe avec les features et la target 'Montant'

# Winsorisation déjà appliquée aux features et à la target
# Transformation log1p de la target
df['Montant_log'] = np.log1p(df['Montant_winsor'])

# Séparation features / target
X = df.drop(columns=['Montant_winsor', 'Montant_log'])
y = df['Montant_log']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensions X_train : {X_train.shape}")
print(f"Dimensions X_test  : {X_test.shape}")
print(f"Dimensions y_train : {y_train.shape}")
print(f"Dimensions y_test  : {y_test.shape}")


KeyError: 'Montant_winsor'

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# Définition des hyperparamètres à tester
param_dist_gb = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.6, 0.7, 0.8, 1.0],
    'max_features': [None, 0.7, 0.8],
    'learning_rate': [0.05, 0.1, 0.2]
}

# RandomizedSearchCV pour fine-tuning rapide
random_search_gb = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=50,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

# Fit sur la target log-transformée
random_search_gb.fit(X_train, y_train)

# Meilleur modèle
best_gb = random_search_gb.best_estimator_

# Prédiction et retour à l'échelle originale
y_pred_log = best_gb.predict(X_test)
y_pred = np.expm1(y_pred_log)  # inverse de log1p

# RMSE sur l'échelle originale
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred))
print(f"Gradient Boosting RMSE fine-tuning sur target log-transformée : {rmse:.4f}")
print("Meilleurs hyperparamètres GB :", random_search_gb.best_params_)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# Transformation log1p sur la target
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Définition de l'espace des hyperparamètres pour Random Forest
param_dist_rf = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.6, 0.8, 1.0, None]
}

# RandomizedSearchCV
rf_log = RandomForestRegressor(random_state=42)
random_search_rf = RandomizedSearchCV(
    estimator=rf_log,
    param_distributions=param_dist_rf,
    n_iter=50,
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# Entraînement
random_search_rf.fit(X_train, y_train_log)
best_rf_log = random_search_rf.best_estimator_

# Prédictions sur la target log-transformée
y_pred_log = best_rf_log.predict(X_test)
y_pred = np.expm1(y_pred_log)  # Retour à l'échelle originale

# Calcul du RMSE
rmse_rf_log = np.sqrt(mean_squared_error(y_test, y_pred))
print("Random Forest RMSE fine-tuning sur target log-transformée :", rmse_rf_log)
print("Meilleurs hyperparamètres RF :", random_search_rf.best_params_)


In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv",sep=";")
df


In [ ]:
# ==============================
# Fonction de prédiction du Montant
# ==============================

import pandas as pd
import numpy as np

def predict_montant(input_features):
    """
    input_features : dict ou DataFrame avec toutes les features utilisées pour le modèle
                     (mêmes colonnes que X_train)
    retourne : prédiction du Montant réel
    """
    # Si input_features est un dictionnaire, le convertir en DataFrame
    if isinstance(input_features, dict):
        input_df = pd.DataFrame([input_features])
    else:
        input_df = input_features.copy()
    
    # Vérifier que toutes les colonnes du modèle sont présentes
    missing_cols = set(X_train.columns) - set(input_df.columns)
    for col in missing_cols:
        input_df[col] = 0.0  # remplir les colonnes manquantes avec 0
    
    input_df = input_df[X_train.columns]  # réordonner les colonnes comme le train
    
    # Prédiction log-transformée
    y_pred_log = best_rf_log.predict(input_df)
    
    # Retour à l'échelle originale
    y_pred = np.expm1(y_pred_log)
    
    return y_pred

# ==============================
# Exemple d'utilisation
# ==============================

# Remplacer ces valeurs par celles que vous voulez prédire
exemple_features = {
    'Echéance': 62.0,
    'Nombre de Titres': 1459.0,
    'Taux': 7.60,
    # toutes les autres colonnes seront automatiquement ajoutées à 0
}

prediction = predict_montant(exemple_features)
print("Montant prédit :", prediction)
